In [ ]:
# Synthetic HR Employee Attrition Dataset

# Import Libraries
import os #no need if using google collab
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

np.random.seed(42)

N = 5000  # number of employees to be changed depending on the required dataset size

# Employee ID generation
emp_id = [f"EMP{str(i).zfill(4)}" for i in range(N)]

# Age generation
age = np.random.normal(38, 10, N).astype(int)
age = np.clip(age, 22, 60)

# Years at Company
years_at_company = np.zeros(N, dtype=int)
for i in range(N):
    max_years = min(age[i]-22, 35)
    years_at_company[i] = np.random.randint(0, max_years + 1)

# Department
departments = np.random.choice(
    ['Sales', 'Engineering', 'HR', 'Marketing', 'Finance', 'Operations', 'IT'],
    size=N,
    p=[0.25, 0.20, 0.08, 0.12, 0.10, 0.15, 0.10]
)

# Job Level
job_level = np.ones(N, dtype=int)
for i in range(N):
    if years_at_company[i] < 2:
        job_level[i] = 1
    elif years_at_company[i] < 5:
        job_level[i] = np.random.choice([1, 2], p=[0.3, 0.7])
    elif years_at_company[i] < 10:
        job_level[i] = np.random.choice([2, 3], p=[0.4, 0.6])
    elif years_at_company[i] < 15:
        job_level[i] = np.random.choice([2, 3, 4], p=[0.2, 0.5, 0.3])
    else:
        job_level[i] = np.random.choice([3, 4, 5], p=[0.3, 0.5, 0.2])

# Monthly Income
base_salary = {
    'Sales': 4500, 'Engineering': 6000, 'HR': 4000,
    'Marketing': 5000, 'Finance': 5500, 'Operations': 4200, 'IT': 5800
}
monthly_income = np.zeros(N, dtype=int)
for i in range(N):
    base = base_salary[departments[i]]
    level_multiplier = 1 + (job_level[i] - 1) * 0.4
    experience_bonus = years_at_company[i] * 150
    noise = np.random.normal(0, 500)
    monthly_income[i] = int(base * level_multiplier + experience_bonus + noise)
monthly_income = np.clip(monthly_income, 3000, 25000)

# Job Satisfaction
job_satisfaction = np.random.choice([1, 2, 3, 4, 5], size=N, p=[0.05, 0.15, 0.35, 0.30, 0.15])

# Work-Life Balance
work_life_balance = np.random.choice([1, 2, 3, 4], size=N, p=[0.10, 0.25, 0.45, 0.20])

# Overtime
overtime_prob = np.where(np.isin(departments, ['Sales', 'Engineering', 'IT']), 0.35, 0.20)
overtime = np.random.binomial(1, overtime_prob)
overtime_display = np.where(overtime==1, 'Yes', 'No')

# Distance from home
distance_from_home = np.random.gamma(2, 5, N)
distance_from_home = np.clip(distance_from_home, 1, 50).round(1)

# Promotions last 5 years
promotion_prob = np.where(years_at_company >= 5, 0.12, 0.05)
promotion = np.random.binomial(1, promotion_prob)
promotion_display = np.where(promotion==1, 'Yes', 'No')

# Performance Rating
performance_rating = np.random.choice([1, 2, 3, 4], size=N, p=[0.05, 0.40, 0.45, 0.10])

# Training hours last year: Poisson around 15
training_hours = np.random.poisson(15, N)
training_hours = np.clip(training_hours, 0, 80)

# Attrition Simulation
attrition_logit = (
    -3.0 +
    0.8 * (job_satisfaction <= 2) +
    0.6 * (work_life_balance <= 2) +
    0.5 * overtime +
    0.4 * (years_at_company < 2) +
    -0.3 * (years_at_company > 10) +
    0.3 * (promotion == 0) * (years_at_company >= 3) +
    -0.4 * (job_level >= 4) +
    0.3 * (distance_from_home > 30) +
    -0.2 * (performance_rating >= 3) +
    0.2 * (monthly_income < 5000)
)
prob_attrition = 1 / (1 + np.exp(-attrition_logit))
attrition = np.random.binomial(1, prob_attrition)


# Build DataFrame
df = pd.DataFrame({
    'EmployeeID': emp_id,
    'Age': age,
    'Department': departments,
    'JobLevel': job_level,
    'YearsAtCompany': years_at_company,
    'MonthlyIncome': monthly_income,
    'JobSatisfaction': job_satisfaction,
    'WorkLifeBalance': work_life_balance,
    'OverTime': overtime_display,
    'DistanceFromHome': distance_from_home,
    'PromotionLast5Years': promotion_display,
    'PerformanceRating': performance_rating,
    'TrainingHoursLastYear': training_hours,
    'Attrition': np.where(attrition==1, 'Yes', 'No')
})

#saves in csv format
os.makedirs("data", exist_ok=True)
csv_path = os.path.join("data", "hr_employee_attrition.csv")
df.to_csv(csv_path, index=False)
print(f"Dataset saved to {csv_path}")

#basic analysis of the dataset
print("\n Dataset Summary")
print(df.describe(include='all'))

# Attrition distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Attrition')
plt.title("Attrition Distribution")
plt.show()

# Attrition by Department
plt.figure(figsize=(8,5))
dept_attrition = df.groupby('Department')['Attrition'].value_counts(normalize=True).unstack()
dept_attrition.plot(kind='bar', stacked=True)
plt.title("Attrition Rate by Department")
plt.ylabel("Proportion")
plt.show()

# Job Satisfaction vs Attrition
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='JobSatisfaction', hue='Attrition')
plt.title("Job Satisfaction vs Attrition")
plt.show()